# Laboratory 1 — Foundations and geometry

**Competencies:** C1 (Understand), C2 (Apply)

By the end of this laboratory you should be able to:

1. trace the chain from applied pressure to bridge output voltage, with correct signs;
2. build a parametric diaphragm geometry and generate a mesh whose element size you control.

Your own geometry is in `instructor/variants/`. Use it, not the reference design, 
for anything you hand in.


## 1. The transduction chain

Pressure deflects the diaphragm. Deflection produces bending stress, largest at the edge 
midpoints. Stress changes the resistance of the piezoresistors through the piezoresistive 
effect. The bridge converts that resistance change into a voltage.

Work through it by hand first. The whole chain is eight lines of algebra, and doing it by 
hand is what lets you tell later whether the simulation is right.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import numpy as np, matplotlib.pyplot as plt
from memslab.plate import Diaphragm, REFERENCE, REFERENCE_PRESSURE

d = REFERENCE          # a = 1000 um, h = 20 um
q = REFERENCE_PRESSURE # 100 kPa

print(f'flexural rigidity D = {d.D:.4e} N m')
print(f'central deflection  = {d.w_max(q)*1e6:.3f} um')
sigma_L, sigma_T = d.edge_stress(q)
print(f'edge stress         = {sigma_L/1e6:.1f} MPa (normal), {sigma_T/1e6:.1f} MPa (parallel)')
long_arm, trans_arm = d.resistance_change(q)
print(f'dR/R                = {long_arm*100:+.2f} %% (long), {trans_arm*100:+.2f} %% (trans)')
print(f'bridge output       = {d.bridge_output(q)*1e3:.1f} mV/V')
print(f'sensitivity         = {d.sensitivity(q)*1e6:.3f} uV/V/Pa')


### Check the model before you trust it

The thin-plate relations hold only while the deflection stays small against the thickness. 
The conventional criterion is $w_{max}/h < 0.2$. Check it every time you change a parameter.


In [ ]:
print(f'w_max/h = {d.w_max(q)/d.h:.4f}')
print('small-deflection theory valid:', d.small_deflection_ok(q))

# TASK: at what pressure does the reference design leave the valid range?
# TASK: repeat for YOUR variant. Does it fail sooner or later than the reference?


## 2. Parametric geometry and meshing

`memslab.geometry` writes GMSH `.geo` sources. Read the generated source before you mesh 
it — you should be able to point at the line that sets each dimension.


In [ ]:
from memslab.geometry import variant_for, geo_source

me = variant_for('YOUR NAME HERE')   # replace with your own name
print(me)
print()
print(geo_source(me, mesh_size_um=40))


### Mesh size against element count

Characteristic length controls how many elements you get, and element count controls how 
long every later run takes. Get a feel for the trade now, because Laboratory 3 asks you to 
choose a mesh deliberately.


In [ ]:
# TASK: mesh your variant at lc = 100, 60, 40, 25, 15 um.
# Record the element count for each, plot count against lc on log axes,
# and state the slope you expect for a surface mesh before you look at the plot.

# gmsh -2 yourname.geo -o yourname.msh    (or use the gmsh Python API)


## Reflection

1. Why does peak stress occur at the edge midpoints and not at the centre or the corners?
2. The two bridge arms respond with opposite sign. What in the physics produces that, and 
   what would the bridge output be if both arms responded the same way?
